In [1]:
import pandas as pd

df = pd.read_csv("../datasets/dice_com-job_us_sample.csv")

df.head()

,advertiserurl,company,employmenttype_jobstatus,jobdescription,jobid,joblocation_address,jobtitle,postdate,shift,site_name,skills,uniq_id
0,https://www.dice.com/jobs/detail/AUTOMATION-TE...,"Digital Intelligence Systems, LLC","C2H Corp-To-Corp, C2H Independent, C2H W2, 3 M...",Looking for Selenium engineers...must have sol...,Dice Id : 10110693,"Atlanta, GA",AUTOMATION TEST ENGINEER,1 hour ago,Telecommuting not available|Travel not required,NaN,SEE BELOW,418ff92580b270ef4e7c14f0ddfc36b4
1,https://www.dice.com/jobs/detail/Information-S...,University of Chicago/IT Services,Full Time,The University of Chicago has a rapidly growin...,Dice Id : 10114469,"Chicago, IL",Information Security Engineer,1 week ago,Telecommuting not available|Travel not required,NaN,"linux/unix, network monitoring, incident respo...",8aec88cba08d53da65ab99cf20f6f9d9
2,https://www.dice.com/jobs/detail/Business-Solu...,"Galaxy Systems, Inc.",Full Time,"GalaxE.SolutionsEvery day, our solutions affec...",Dice Id : CXGALXYS,"Schaumburg, IL",Business Solutions Architect,2 weeks ago,Telecommuting not available|Travel not required,NaN,"Enterprise Solutions Architecture, business in...",46baa1f69ac07779274bcd90b85d9a72
3,https://www.dice.com/jobs/detail/Java-Develope...,TransTech LLC,Full Time,Java DeveloperFull-time/direct-hireBolingbrook...,Dice Id : 10113627,"Bolingbrook, IL","Java Developer (mid level)- FT- GREAT culture,...",2 weeks ago,Telecommuting not available|Travel not required,NaN,Please see job description,3941b2f206ae0f900c4fba4ac0b18719
4,https://www.dice.com/jobs/detail/DevOps-Engine...,Matrix Resources,Full Time,Midtown based high tech firm has an immediate ...,Dice Id : matrixga,"Atlanta, GA",DevOps Engineer,48 minutes ago,Telecommuting not available|Travel not required,NaN,"Configuration Management, Developer, Linux, Ma...",45efa1f6bc65acc32bbbb953a1ed13b7


In [2]:
print(df.columns)

print(df.info())

Index(['advertiserurl', 'company', 'employmenttype_jobstatus',
       'jobdescription', 'jobid', 'joblocation_address', 'jobtitle',
       'postdate', 'shift', 'site_name', 'skills', 'uniq_id'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22000 entries, 0 to 21999
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   advertiserurl             22000 non-null  object
 1   company                   21950 non-null  object
 2   employmenttype_jobstatus  21770 non-null  object
 3   jobdescription            22000 non-null  object
 4   jobid                     22000 non-null  object
 5   joblocation_address       21997 non-null  object
 6   jobtitle                  22000 non-null  object
 7   postdate                  22000 non-null  object
 8   shift                     21643 non-null  object
 9   site_name                 3490 non-null   object
 10  skills               

In [3]:
df = df[[
    "jobtitle",
    "skills",
    "company"
]]

df = df.dropna()

df.head()

,jobtitle,skills,company
0,AUTOMATION TEST ENGINEER,SEE BELOW,"Digital Intelligence Systems, LLC"
1,Information Security Engineer,"linux/unix, network monitoring, incident respo...",University of Chicago/IT Services
2,Business Solutions Architect,"Enterprise Solutions Architecture, business in...","Galaxy Systems, Inc."
3,"Java Developer (mid level)- FT- GREAT culture,...",Please see job description,TransTech LLC
4,DevOps Engineer,"Configuration Management, Developer, Linux, Ma...",Matrix Resources


In [4]:
df["combined_features"] = (
    df["jobtitle"] + " " +
    df["skills"]
)

df.head()

,jobtitle,skills,company,combined_features
0,AUTOMATION TEST ENGINEER,SEE BELOW,"Digital Intelligence Systems, LLC",AUTOMATION TEST ENGINEER SEE BELOW
1,Information Security Engineer,"linux/unix, network monitoring, incident respo...",University of Chicago/IT Services,"Information Security Engineer linux/unix, netw..."
2,Business Solutions Architect,"Enterprise Solutions Architecture, business in...","Galaxy Systems, Inc.",Business Solutions Architect Enterprise Soluti...
3,"Java Developer (mid level)- FT- GREAT culture,...",Please see job description,TransTech LLC,"Java Developer (mid level)- FT- GREAT culture,..."
4,DevOps Engineer,"Configuration Management, Developer, Linux, Ma...",Matrix Resources,"DevOps Engineer Configuration Management, Deve..."


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(
    df["combined_features"]
)

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

In [7]:
def recommend_jobs(skill_input):

    input_vector = tfidf.transform([skill_input])

    similarity = cosine_similarity(
        input_vector,
        tfidf_matrix
    )

    similar_jobs = similarity.argsort()[0][-5:]

    return df.iloc[similar_jobs][[
        "jobtitle",
        "company",
        "skills"
    ]]

In [8]:
recommend_jobs(
    "Python Machine Learning Flask SQL"
)

,jobtitle,company,skills
13735,Mid-Level Python Developer,Smart Synergies,"Python, Django, Flask"
12336,Post-Doctoral Associate - Virtualization and M...,AgreeYa Solutions,"Ph.D. Degree, Machine Learning and python"
16429,Software Engineer C++,Prosum,c++ C machine learning python aws linux machin...
19971,Jr Level Software Engineer Opportunity - Machi...,LexisNexis,"data modeling, Machine learning"
14791,"Back End Developer - Python, SQL, Android, iOS...",CyberCoders,"Python, SQL, Android, Objective-C, Machine Lea..."
